In [1]:
# preprocess.ipynb
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, isnan, isnull, trim, regexp_replace, split, size
from pyspark.sql.types import *
import os

# 创建SparkSession（添加MySQL驱动路径）
spark = SparkSession.builder \
    .appName("MovieDataPreprocess") \
    .master("local[*]") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.driver.extraClassPath", "./mysql-connector-java-8.0.23.jar") \
    .getOrCreate()

# 设置日志级别
spark.sparkContext.setLogLevel("WARN")

# 数据文件路径（假设放在本地data/raw目录）
BASE_PATH = "./data/raw/"
OUTPUT_HDFS = "./data/movie/clean"

In [1]:
import os
import subprocess

# ========== 1. 环境变量强制配置（路径已全部修正） ==========
JAVA_HOME = r"D:\JDK8"                # 匹配你实际的 JDK 根目录
HADOOP_HOME = r"E:\hadoop\hadoop-3.3.5"  # 匹配你实际的 Hadoop 根目录

os.environ["JAVA_HOME"] = JAVA_HOME
os.environ["HADOOP_HOME"] = HADOOP_HOME
# 强制 JVM 使用 IPv4，解决 Windows 监听地址不匹配问题
os.environ["JAVA_TOOL_OPTIONS"] = "-Djava.net.preferIPv4Stack=true"

# 把 Java 和 Hadoop 的可执行目录加入系统 PATH
java_bin = os.path.join(JAVA_HOME, "bin")
hadoop_bin = os.path.join(HADOOP_HOME, "bin")
os.environ["PATH"] = f"{java_bin};{hadoop_bin};" + os.environ.get("PATH", "")

# 环境自检
print("=== 环境自检 ===")
java_ver = subprocess.run(["java", "-version"], capture_output=True, text=True)
print(f"Java 版本: {java_ver.stderr.strip()}")
winutils_ver = subprocess.run(["winutils", "version"], capture_output=True, text=True)
print("winutils: 配置正常")
print("================")

# ========== 2. 创建 SparkSession ==========
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master("local[*]")
         .appName("Movie Analysis")
         # Windows 必加：强制绑定本地 IPv4
         .config("spark.driver.host", "127.0.0.1")
         .config("spark.driver.bindAddress", "127.0.0.1")
         # 固定端口，降低防火墙拦截概率
         .config("spark.driver.port", "40400")
         .config("spark.ui.port", "4040")
         # 超时与 Worker 配置
         .config("spark.python.worker.timeout", "600")
         .config("spark.network.timeout", "600s")
         .config("spark.python.worker.reuse", "false")
         .getOrCreate())

print("\n✅ SparkSession 创建成功")
print(f"Spark 版本: {spark.version}")
print(f"Spark UI 地址: {spark.sparkContext.uiWebUrl}")

# ========== 3. 读取全部数据 ==========
BASE_PATH = "E:/movie_analysiz_01/test02/data/raw/"

movies_raw = spark.read.option("header", True) \
    .option("inferSchema", True) \
    .option("escape", "\"") \
    .option("multiline", True) \
    .csv(BASE_PATH + "movies.csv")

ratings_raw = spark.read.option("header", True) \
    .option("inferSchema", True) \
    .csv(BASE_PATH + "ratings.csv")

users_raw = spark.read.option("header", True) \
    .option("inferSchema", True) \
    .csv(BASE_PATH + "users.csv")

comments_raw = spark.read.option("header", True) \
    .option("inferSchema", True) \
    .option("escape", "\"") \
    .option("multiline", True) \
    .csv(BASE_PATH + "comments.csv")

person_raw = spark.read.option("header", True) \
    .option("inferSchema", True) \
    .option("escape", "\"") \
    .csv(BASE_PATH + "person.csv")

# ========== 4. 输出验证结果 ==========
print("\n📊 原始数据行数：")
print(f"movies: {movies_raw.count()}")
print(f"ratings: {ratings_raw.count()}")
print(f"users: {users_raw.count()}")
print(f"comments: {comments_raw.count()}")
print(f"person: {person_raw.count()}")

print("\n🎬 movies 前5行：")
movies_raw.show(5)

=== 环境自检 ===
Java 版本: java version "1.8.0_181"
Java(TM) SE Runtime Environment (build 1.8.0_181-b13)
Java HotSpot(TM) 64-Bit Server VM (build 25.181-b13, mixed mode)
Picked up JAVA_TOOL_OPTIONS: -Djava.net.preferIPv4Stack=true
winutils: 配置正常

✅ SparkSession 创建成功
Spark 版本: 3.5.3
Spark UI 地址: http://127.0.0.1:4040

📊 原始数据行数：
movies: 140502
ratings: 4169420
users: 639128
comments: 4428475
person: 72959

🎬 movies 前5行：
+--------+--------------+--------------+---------------------------------+-----+-----------+------------+------------+---------+-------+------------------------+----+-------------+---------------+------------+---------+-------------------------------------+---------------------------------+------+--------------------------+--------------+
|MOVIE_ID|          NAME|         ALIAS|                           ACTORS|COVER|  DIRECTORS|DOUBAN_SCORE|DOUBAN_VOTES|   GENRES|IMDB_ID|               LANGUAGES|MINS|OFFICIAL_SITE|        REGIONS|RELEASE_DATE|     SLUG|                      

In [8]:
from pyspark.sql import functions as F

def clean_movies(df):
    """
    清洗电影数据movies.csv（22个大写原始字段）
    清洗规则：
    1. 不再过滤无评分影片，全量保留所有电影记录；
    2. 新增标记字段SCORE_FLAG：DOUBAN_SCORE>0为valid，空/0标记UnScore；
    3. 全量22字段缺失值填充：文本字段填充Unknown，数字填充0；
    4. 年份转为整型衍生字段YEAR_INT；
    5. 标准化GENRES、REGIONS、LANGUAGES，清除多余空格；
    6. 豆瓣评分统一保留1位小数；
    7. 删除无业务分析价值冗余字段 SLUG/IMDB_ID/OFFICIAL_SITE/COVER/ALIAS
    """
    # 1. 取消过滤无评分电影，所有数据全部保留
    df_clean = df

    # 2. 新增评分标记字段，区分有效评分与无评分影片
    df_clean = df_clean.withColumn(
        "SCORE_FLAG",
        F.when(F.col("DOUBAN_SCORE") > 0, F.lit("valid"))
        .otherwise(F.lit("UnScore"))
    )

    # 3. 全字段缺失值填充：所有文本空值统一填 Unknown，数值填0
    df_clean = df_clean.fillna({
        "MOVIE_ID": "Unknown",
        "NAME": "Unknown",
        "ALIAS": "Unknown",
        "ACTORS": "Unknown",
        "COVER": "Unknown",
        "DIRECTORS": "Unknown",
        "DOUBAN_SCORE": 0.0,
        "DOUBAN_VOTES": 0,
        "GENRES": "Unknown",
        "IMDB_ID": "Unknown",
        "LANGUAGES": "Unknown",
        "MINS": 0,
        "OFFICIAL_SITE": "Unknown",
        "REGIONS": "Unknown",
        "RELEASE_DATE": "Unknown",
        "SLUG": "Unknown",
        "STORYLINE": "Unknown",
        "TAGS": "Unknown",
        "YEAR": "Unknown",
        "ACTOR_IDS": "Unknown",
        "DIRECTOR_IDS": "Unknown"
    })

    # 4. 年份清洗，生成整型年份字段 YEAR_INT
    df_clean = df_clean.withColumn("YEAR_INT",
        F.when(F.col("YEAR").cast("int").isNotNull(), F.col("YEAR").cast("int"))
        .otherwise(0)
    )

    # 5. 电影类型清洗：去除全部多余空格
    df_clean = df_clean.withColumn("GENRES_CLEAN",
        F.regexp_replace(F.trim(F.col("GENRES")), "\\s+", "")
    )

    # 6. 地区字段标准化
    df_clean = df_clean.withColumn("REGIONS_CLEAN",
        F.regexp_replace(F.trim(F.col("REGIONS")), "\\s+", "")
    )

    # 7. 语言字段标准化（修复原代码笔误，REGIONS改为LANGUAGES）
    df_clean = df_clean.withColumn("LANGUAGES_CLEAN",
        F.regexp_replace(F.trim(F.col("LANGUAGES")), "\\s+", "")
    )

    # 8. 豆瓣评分统一保留1位小数
    df_clean = df_clean.withColumn("DOUBAN_SCORE",
        F.round(F.col("DOUBAN_SCORE"), 1)
    )

    # 9. 删除无用冗余字段
    cols_to_drop = ["SLUG", "IMDB_ID", "OFFICIAL_SITE", "COVER", "ALIAS"]
    for c in cols_to_drop:
        if c in df_clean.columns:
            df_clean = df_clean.drop(c)

    return df_clean

# 调用清洗函数
movies_clean = clean_movies(movies_raw)

# 统计总数据、正常有效评分数据、无评分异常数据
total_movie = movies_clean.count()
valid_movie = movies_clean.filter(F.col("SCORE_FLAG") == "valid").count()
abnormal_movie = movies_clean.filter(F.col("SCORE_FLAG") == "UnScore").count()

# 打印清洗结果与异常数据数量
print("==========电影数据清洗统计结果==========")
print(f"清洗后电影总数据行数: {total_movie} 行")
print(f"有效评分正常数据(参与模型/推荐): {valid_movie} 条")
print(f"无评分异常数据(UnScore，仅留存不参与计算): {abnormal_movie} 条")

# 打印表结构
movies_clean.printSchema()

==========电影数据清洗统计结果==========
清洗后电影总数据行数: 140502 行
有效评分正常数据(参与模型/推荐): 25968 条
无评分异常数据(UnScore，仅留存不参与计算): 114534 条
root
 |-- MOVIE_ID: integer (nullable = true)
 |-- NAME: string (nullable = false)
 |-- ACTORS: string (nullable = false)
 |-- DIRECTORS: string (nullable = false)
 |-- DOUBAN_SCORE: double (nullable = true)
 |-- DOUBAN_VOTES: double (nullable = false)
 |-- GENRES: string (nullable = false)
 |-- LANGUAGES: string (nullable = false)
 |-- MINS: double (nullable = false)
 |-- REGIONS: string (nullable = false)
 |-- RELEASE_DATE: date (nullable = true)
 |-- STORYLINE: string (nullable = false)
 |-- TAGS: string (nullable = false)
 |-- YEAR: double (nullable = true)
 |-- ACTOR_IDS: string (nullable = false)
 |-- DIRECTOR_IDS: string (nullable = false)
 |-- SCORE_FLAG: string (nullable = false)
 |-- YEAR_INT: integer (nullable = true)
 |-- GENRES_CLEAN: string (nullable = false)
 |-- REGIONS_CLEAN: string (nullable = false)
 |-- LANGUAGES_CLEAN: string (nullable = false)



In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
import os

# 初始化Spark
spark = SparkSession.builder \
    .appName("MovieDataPreprocess") \
    .master("local[*]") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.driver.extraClassPath", "./mysql-connector-java-8.0.23.jar") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

# 路径配置
BASE_PATH = r"E:\movie_analysiz_01\test02\data\raw"
OUTPUT_HDFS = r"E:\movie_analysiz_01\test02\output\movie\clean"

# 评分清洗函数，仅保留基础清洗逻辑，无额外拓展功能
def clean_ratings(df):
    # 过滤有效评分、用户ID、电影ID不为空
    df_clean = df.filter(
        (col("RATING").between(1, 5)) &
        (col("USER_MD5").isNotNull()) &
        (col("MOVIE_ID").isNotNull())
    )
    # 时间字段转为时间戳
    df_clean = df_clean.withColumn("RATING_TIME",
        when(col("RATING_TIME").isNotNull(), col("RATING_TIME").cast("timestamp"))
        .otherwise(None)
    )
    return df_clean

# 拼接文件路径
csv_path = os.path.join(BASE_PATH, "ratings.csv")
print("读取文件完整路径：", csv_path)

# 读取原始评分csv
ratings_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(csv_path)

# 执行清洗
ratings_clean = clean_ratings(ratings_raw)
print(f"清洗后评分数据: {ratings_clean.count()} 行")

读取文件完整路径： E:\movie_analysiz_01\test02\data\raw\ratings.csv
清洗后评分数据: 4169420 行


In [5]:
def clean_users(df):
    """用户数据清洗，填充空昵称"""
    df_clean = df.filter(col("USER_MD5").isNotNull())
    df_clean = df_clean.fillna({"USER_NICKNAME": "匿名用户"})
    return df_clean

users_clean = clean_users(users_raw)
print(f"清洗后用户数据: {users_clean.count()} 行")

清洗后用户数据: 639128 行


In [6]:
def clean_comments(df):
    """
    评论数据清洗：
    - 过滤空评论内容、空用户ID、空电影ID
    - 评论时间转为 Timestamp 类型（无效值置为 null）
    - 赞同数填充为 0（若为 null）
    - 评分字段转为整数（若存在且非空）
    """
    df_clean = df.filter(
        (col("CONTENT").isNotNull()) &
        (col("CONTENT") != "") &
        (col("USER_MD5").isNotNull()) &
        (col("MOVIE_ID").isNotNull())
    )

    # 评论时间转换（原字段可能是字符串）
    df_clean = df_clean.withColumn(
        "COMMENT_TIME",
        when(col("COMMENT_TIME").isNotNull(), col("COMMENT_TIME").cast("timestamp"))
        .otherwise(None)
    )

    # 赞同数填充 0（原字段可能是整数或字符串，inferSchema 通常能识别）
    df_clean = df_clean.fillna({"VOTES": 0})

    # 如果 RATING 字段存在，确保类型为整数（可选）
    if "RATING" in df.columns:
        df_clean = df_clean.withColumn(
            "RATING",
            when(col("RATING").isNotNull(), col("RATING").cast(IntegerType()))
            .otherwise(None)
        )

    return df_clean

# 4. 执行清洗
comments_clean = clean_comments(comments_raw)

# 5. 查看结果
print(f"清洗后评论数据: {comments_clean.count()} 行")
comments_clean.printSchema()
comments_clean.show(5, truncate=False)

清洗后评论数据: 4428475 行
root
 |-- COMMENT_ID: integer (nullable = true)
 |-- USER_MD5: string (nullable = true)
 |-- MOVIE_ID: integer (nullable = true)
 |-- CONTENT: string (nullable = true)
 |-- VOTES: integer (nullable = false)
 |-- COMMENT_TIME: timestamp (nullable = true)
 |-- RATING: integer (nullable = true)

+----------+--------------------------------+--------+-----------------------------------------------------------------------------------------------+-----+-------------------+------+
|COMMENT_ID|USER_MD5                        |MOVIE_ID|CONTENT                                                                                        |VOTES|COMMENT_TIME       |RATING|
+----------+--------------------------------+--------+-----------------------------------------------------------------------------------------------+-----+-------------------+------+
|1359352573|0ab7e3efacd56983f16503572d2b9915|5113101 |480p，画质不高，黑白，y                                                                   

In [7]:
def clean_person(df):
    """人物数据清洗：过滤空姓名，填充未知性别"""
    df_clean = df.filter(col("NAME").isNotNull() & (col("NAME") != ""))
    df_clean = df_clean.fillna({"SEX": "未知", "PROFESSION": "未知"})
    return df_clean

person_clean = clean_person(person_raw)
print(f"清洗后人物数据: {person_clean.count()} 行")

清洗后人物数据: 72959 行


In [9]:
# 定义HDFS输出目录（我们直接使用项目中的相对路径保存）
hdfs_base = "./data/movie/clean"

# 保存各表为CSV格式（含header）
movies_clean.write.mode("overwrite") \
    .option("header", "true") \
    .csv(f"{hdfs_base}/movies")

ratings_clean.write.mode("overwrite") \
    .option("header", "true") \
    .csv(f"{hdfs_base}/ratings")

users_clean.write.mode("overwrite") \
    .option("header", "true") \
    .csv(f"{hdfs_base}/users")

comments_clean.write.mode("overwrite") \
    .option("header", "true") \
    .csv(f"{hdfs_base}/comments")

person_clean.write.mode("overwrite") \
    .option("header", "true") \
    .csv(f"{hdfs_base}/person")

print("所有清洗数据已保存到HDFS")

所有清洗数据已保存到HDFS


In [10]:
# 路径配置
hdfs_base = r"E:\movie_analysiz_01\test02\output\movie\clean"

# 通用保存函数
def save_to_parquet(df, table_name):
    df.write.mode("overwrite").parquet(f"{hdfs_base}/{table_name}_parquet")
    print(f"✅ {table_name}_parquet 写入完成")

# 批量一键保存所有清洗后的表
save_to_parquet(movies_clean, "movies")
save_to_parquet(ratings_clean, "ratings")
save_to_parquet(users_clean, "users")
save_to_parquet(person_clean, "person")
save_to_parquet(comments_clean, "comments")

print("\n===== 全部5张表Parquet文件导出完毕 =====")

✅ movies_parquet 写入完成
✅ ratings_parquet 写入完成
✅ users_parquet 写入完成
✅ person_parquet 写入完成
✅ comments_parquet 写入完成

===== 全部5张表Parquet文件导出完毕 =====


In [1]:
# import_to_mysql.py
from pyspark.sql import SparkSession
import os

# MySQL连接配置
MYSQL_CONFIG = {
    "url": "jdbc:mysql://localhost:3306/movie_db?useSSL=false&serverTimezone=Asia/Shanghai&allowPublicKeyRetrieval=true",
    "user": "root",
    "password": "M20054921",
    "driver": "com.mysql.cj.jdbc.Driver"
}

def write_to_mysql(df, table, mode="overwrite"):
    """将DataFrame写入MySQL表"""
    df.write \
        .format("jdbc") \
        .option("url", MYSQL_CONFIG["url"]) \
        .option("dbtable", table) \
        .option("user", MYSQL_CONFIG["user"]) \
        .option("password", MYSQL_CONFIG["password"]) \
        .option("driver", MYSQL_CONFIG["driver"]) \
        .mode(mode) \
        .save()
    print(f"✅ 数据已写入 {table}，行数: {df.count()}")

if __name__ == "__main__":
    # -------------------------- 修复驱动加载 --------------------------
    # 1. 确认jar路径，核对文件是否真实存在
    jar_full_path = r"E:\movie_analysiz_01\test01\output\mysql-connector-java-8.0.23.jar"
    # Jupyter环境变量前置配置（必须放在Spark初始化之前）
    os.environ["PYSPARK_SUBMIT_ARGS"] = f"--jars '{jar_full_path}' pyspark-shell"

    # 2. Spark会话初始化，三重配置驱动路径
    spark = SparkSession.builder \
        .appName("ImportToMySQL") \
        .master("local[*]") \
        .config("spark.jars", jar_full_path) \
        .config("spark.driver.extraClassPath", jar_full_path) \
        .config("spark.executor.extraClassPath", jar_full_path) \
        .getOrCreate()
    spark.sparkContext.setLogLevel("WARN")

    # CSV数据根目录
    base_path = "E:/movie_analysiz_01/test02/output/data/movie/clean"

    print("数据根目录：", base_path)
    print("MySQL驱动jar路径：", jar_full_path)

    # 读取CSV分片文件夹
    movies = spark.read.option("header", True).csv(f"{base_path}/movies")
    ratings = spark.read.option("header", True).csv(f"{base_path}/ratings")
    users = spark.read.option("header", True).csv(f"{base_path}/users")
    comments = spark.read.option("header", True).csv(f"{base_path}/comments")
    person = spark.read.option("header", True).csv(f"{base_path}/person")

    # ===================== 字段映射 =====================
    # 1.电影表 movie
    movies = movies.selectExpr(
        "MOVIE_ID as movie_id",
        "NAME as name",
        "GENRES_CLEAN as genres",
        "DIRECTORS as directors",
        "ACTORS as actors",
        "DOUBAN_SCORE as douban_score",
        "DOUBAN_VOTES as douban_votes",
        "YEAR_INT as year",
        "RELEASE_DATE as release_date",
        "LANGUAGES_CLEAN as language",
        "REGIONS_CLEAN as region",
        "STORYLINE as storyline",
        "MINS as mins",
        "TAGS as tags",
        "ACTOR_IDS as actor_ids",
        "DIRECTOR_IDS as director_ids",
        "SCORE_FLAG as SCORE_FLAG"
    )

    # 2.评分表 rating
    ratings = ratings.selectExpr(
        "RATING_ID as RATING_ID",
        "USER_MD5 as USER_MD5",
        "MOVIE_ID as MOVIE_ID",
        "RATING as RATING",
        "RATING_TIME as RATING_TIME"
    )

    # 3.用户表 user
    users = users.selectExpr(
        "USER_MD5 as USER_MD5",
        "USER_NICKNAME as USER_NICKNAME"
    )

    # 4.评论表 comment
    comments = comments.selectExpr(
        "COMMENT_ID as COMMENT_ID",
        "USER_MD5 as USER_MD5",
        "MOVIE_ID as MOVIE_ID",
        "CONTENT as CONTENT",
        "VOTES as VOTES",
        "COMMENT_TIME as COMMENT_TIME",
        "RATING as RATING"
    )

    # 5.影人表 person
    person = person.selectExpr(
        "PERSON_ID as PERSON_ID",
        "NAME as NAME",
        "SEX as SEX",
        "NAME_EN as NAME_EN",
        "NAME_ZH as NAME_ZH",
        "BIRTH as BIRTH",
        "BIRTHPLACE as BIRTHPLACE",
        "CONSTELLATORY as CONSTELLATORY",
        "PROFESSION as PROFESSION",
        "BIOGRAPHY as BIOGRAPHY"
    )

    # 批量写入MySQL
    print("\n===== 开始导入5张CSV数据表到MySQL =====")
    write_to_mysql(movies, "movie")
    write_to_mysql(ratings, "rating")
    write_to_mysql(users, "user")
    write_to_mysql(comments, "comment")
    write_to_mysql(person, "person")

    print("\n🎉 全部CSV数据导入MySQL完成！")
    spark.stop()

数据根目录： E:/movie_analysiz_01/test02/output/data/movie/clean
MySQL驱动jar路径： E:\movie_analysiz_01\test01\output\mysql-connector-java-8.0.23.jar

===== 开始导入5张CSV数据表到MySQL =====
✅ 数据已写入 movie，行数: 140502
✅ 数据已写入 rating，行数: 4169420
✅ 数据已写入 user，行数: 639128
✅ 数据已写入 comment，行数: 4649354
✅ 数据已写入 person，行数: 72959

🎉 全部CSV数据导入MySQL完成！


In [2]:
# first_analysis.py
from pyspark.sql.functions import count, avg, sum as spark_sum, round, col
from pyspark.sql import SparkSession

# MySQL连接配置
MYSQL_CONFIG = {
    "url": "jdbc:mysql://localhost:3306/movie_db?useSSL=false&serverTimezone=Asia/Shanghai",
    "user": "root",
    "password": "M20054921",
    "driver": "com.mysql.cj.jdbc.Driver"
}


spark = SparkSession.builder \
    .appName("FirstAnalysis") \
    .master("local[*]") \
    .config("spark.driver.extraClassPath", "./mysql-connector-java-8.0.23.jar") \
    .getOrCreate()

# 读取清洗后的电影数据
movies = spark.read.option("header", True).csv("./data/movie/clean/movies")

# 按年份统计电影数量、平均评分、总投票数
year_stats = movies.filter(col("YEAR_INT") > 0) \
    .groupBy("YEAR_INT") \
    .agg(
        count("MOVIE_ID").alias("movie_count"),
        round(avg("DOUBAN_SCORE"), 2).alias("avg_score"),
        spark_sum("DOUBAN_VOTES").alias("total_votes")
    ) \
    .orderBy("YEAR_INT")

year_stats.show(20)

# 将结果写入MySQL统计表
from pyspark.sql.functions import lit, col, when, to_json, struct

# 构造统计结果DataFrame（格式适配statistics_result表）
stats_df = year_stats.select(
    lit("year_trend").alias("stat_type"),
    col("YEAR_INT").cast("string").alias("stat_key"),
    col("avg_score").cast("string").alias("stat_value"),
    col("movie_count").alias("stat_count"),
    lit(None).cast("decimal(10,2)").alias("stat_percentage"),
    to_json(struct("total_votes")).alias("extra_data")
)

# 写入MySQL
stats_df.write \
    .format("jdbc") \
    .option("url", MYSQL_CONFIG["url"]) \
    .option("dbtable", "movie_stats") \
    .option("user", MYSQL_CONFIG["user"]) \
    .option("password", MYSQL_CONFIG["password"]) \
    .option("driver", MYSQL_CONFIG["driver"]) \
    .mode("append") \
    .save()

print("年度统计结果已写入MySQL")
spark.stop()

+--------+-----------+---------+-----------+
|YEAR_INT|movie_count|avg_score|total_votes|
+--------+-----------+---------+-----------+
|    1873|          1|      0.0|        0.0|
|    1875|          1|      0.0|        0.0|
|    1898|         10|      0.0|        0.0|
|    1900|          3|     2.57|       46.0|
|    1901|          1|      0.0|        0.0|
|    1904|          1|      0.0|        0.0|
|    1905|          2|      0.0|        0.0|
|    1906|          3|      0.0|        0.0|
|    1907|          6|     1.28|       93.0|
|    1908|          8|      0.0|        0.0|
|    1909|         16|      0.0|        0.0|
|    1910|         21|      0.0|        0.0|
|    1911|         24|     0.34|      456.0|
|    1912|         27|      0.0|        0.0|
|    1913|         40|     0.77|      584.0|
|    1914|         67|     0.12|      389.0|
|    1915|        105|     0.37|     6153.0|
|    1916|        107|     0.22|     5796.0|
|    1917|        127|      0.3|      600.0|
|    1918|